# Pipeline:  

Text --> Preprocessing --> Embeddings (WOrd2Vec) --> Embedding Matrix --> RNN --> Sentiment Prediction

In [2]:
!pip install gensim tensorflow nltk pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.7 MB/s eta 0:00:00


# Dataset Link:
https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [4]:
import pandas as pd

In [5]:
df = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [6]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [7]:
texts = df['review']
labels = df['sentiment']

labels = labels.map({'positive':1, 'negative':0})

In [8]:
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [9]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [10]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [11]:
stop_words = stopwords.words('english')

def preprocess(text):
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)

    # Remove mentions & hashtags
    text = re.sub(r'\@\w+|\#','', text)

    # Remove punctuation & numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    return tokens

In [12]:
# apply preprocessing
tokenized_texts = texts.apply(preprocess)

"I love this movie"

↓ tokens  

["i","love","this","movie"]


# train Word2Vec model

In [13]:
from gensim.models import Word2Vec

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

good → [0.21, -0.44, 0.82, ...]  
bad → [-0.52, 0.12, -0.33, ...]

In [15]:
# convert words into integer sequence
# Neural network cannot process words directly
# so we convert words to Integers IDs

tokenizer = Tokenizer()

tokenizer.fit_on_texts(tokenized_texts)

sequences = tokenizer.texts_to_sequences(tokenized_texts)

word_index = tokenizer.word_index

i → 4  
love → 15  
this → 8  
movie → 12  

[4,15,8,12]

In [16]:
# Neural networks requires samel length inputs
max_len = 100

X = pad_sequences(sequences, maxlen=max_len)
y = labels.values

[0,0,0,0,4,15,8,12]

In [17]:
import numpy as np

In [18]:
# Now we convert Word2Vec embeddings to neural network

embedding_dim = 100
vocab_size = len(word_index) + 1

embedding_matrix = np.zeros((vocab_size, embedding_dim))

In [19]:
for word, i in word_index.items():

    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

movie → vector from Word2Vec  
good → vector from Word2Vec

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

from sklearn.model_selection import train_test_split

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    )
)

model.add(SimpleRNN(64))

model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


The embedding layer looks up the vector for each index

4  → [0.21, -0.11, 0.67, ...]  
15 → [0.32, 0.08, -0.55, ...]  
8  → [-0.44, 0.29, 0.31, ...]  
12 → [0.52, -0.71, 0.11, ...]  

In [24]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [25]:
model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.6467 - loss: 0.6321 - val_accuracy: 0.6788 - val_loss: 0.5927
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 28s 22ms/step - accuracy: 0.7334 - loss: 0.5383 - val_accuracy: 0.6581 - val_loss: 0.6171
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 42s 23ms/step - accuracy: 0.7002 - loss: 0.5772 - val_accuracy: 0.7794 - val_loss: 0.4959
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 28s 22ms/step - accuracy: 0.7643 - loss: 0.5086 - val_accuracy: 0.7855 - val_loss: 0.4865
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 41s 23ms/step - accuracy: 0.7353 - loss: 0.5377 - val_accuracy: 0.6559 - val_loss: 0.6085


In [26]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.6589 - loss: 0.6056
Test Accuracy: 0.6559000015258789


In [27]:
def predict_sentiment(text):

    tokens = preprocess(text)

    seq = tokenizer.texts_to_sequences([tokens])

    padded = pad_sequences(seq, maxlen=max_len)

    prediction = model.predict(padded)

    if prediction > 0.5:
        return "Positive"
    else:
        return "Negative"

In [28]:
predict_sentiment("This movie was fantastic")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step


'Positive'